# Warpsmith on a cloud GPU

Works on Google Colab (Runtime > Change runtime type > T4 GPU) and Kaggle (Settings > Accelerator > GPU, Internet on).

Milestone 1: run the evaluation harness on hand-written matmul kernels. No LLM yet.

In [ ]:
!nvidia-smi
!nvcc --version

In [ ]:
import os

REPO_URL = "https://github.com/SupreetTadeparti/warpsmith.git"
WORKDIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "/content"
os.chdir(WORKDIR)
if os.path.exists("warpsmith"):
    !git -C warpsmith pull --ff-only
else:
    !git clone {REPO_URL}
os.chdir(f"{WORKDIR}/warpsmith")
!pip install -q -e ".[dev]"

In [ ]:
!python -m warpsmith info

In [ ]:
# GPU tests run here (they skip on machines without nvcc), including the red-team suite.
!python -m pytest -q -rs

In [ ]:
!python -m warpsmith eval matmul problems/matmul/naive.cu --no-compare

In [ ]:
# Tiled kernel, compared against the naive kernel and cuBLAS (via torch.matmul).
!mkdir -p runs
!python -m warpsmith eval matmul examples/matmul/tiled.cu --json runs/tiled.json

In [ ]:
# Watch the harness catch individual cheats.
!python -m warpsmith eval matmul tests/redteam/matmul/pointer_cache.cu --no-compare
!python -m warpsmith eval matmul tests/redteam/matmul/legacy_stream.cu --no-compare
!python -m warpsmith eval matmul tests/redteam/matmul/oob_far.cu --sanitize --no-bench

## Coming in milestone 3: the agent

Add your UF NaviGator key as a secret named `NAVIGATOR_API_KEY` (Colab: key icon in the sidebar; Kaggle: Add-ons > Secrets). Never paste it into a cell.